# Фиче-инжиниринг: признаки формы

В этом ноутбуке к таблицам результатов гонок добавляются **признаки формы** пилотов и команд.

**Главное правило против утечки данных (data leakage):**
каждый признак формы считается **только по гонкам, которые прошли РАНЬШЕ** текущей.
Поэтому везде сначала сортируем по дате `date_start`, а затем при расчёте делаем
сдвиг `shift(1)` — он выкидывает текущую гонку из её собственного расчёта, чтобы
модель не "подсматривала" результат, который ещё не известен на момент старта.


In [1]:
import os
import numpy as np
import pandas as pd

# Где лежат сырые csv. Пробуем несколько вариантов, т.к. ноутбук можно
# запускать из разных папок (data_prep/ или data/).
def find_raw_dir():
    candidates = ["raw", "../data/raw", "data/raw"]
    for path in candidates:
        if os.path.isdir(path):
            return path
    raise FileNotFoundError("Не нашёл папку raw с сырыми csv")

RAW = find_raw_dir()
print("RAW =", RAW)

RAW = ../data/raw


## Сборка таблиц `race` и `quali`

Если таблицы уже собраны в памяти ноутбука — используем их.
Иначе собираем `race` и `quali` из сырых файлов в `raw/`:
результаты + пилоты + (для гонок) телеметрия кругов + погода + сессии + трассы.
Ключ объединения — `session_key` и `driver_number`.


In [2]:
# Колонки сессий и трасс, которые добавляем к результатам
SESSION_COLS = ["session_key", "circuit_short_name", "country_name", "year", "date_start"]
MEETING_COLS = ["meeting_key", "circuit_type"]


def build_race_table(raw):
    results  = pd.read_csv(f"{raw}/race_results.csv")
    drivers  = pd.read_csv(f"{raw}/drivers.csv")
    laps     = pd.read_csv(f"{raw}/laps_agg.csv")
    weather  = pd.read_csv(f"{raw}/weather.csv")
    sessions = pd.read_csv(f"{raw}/sessions.csv")
    meetings = pd.read_csv(f"{raw}/meetings.csv")

    df = results.merge(drivers, on=["session_key", "driver_number"], how="left")
    df = df.merge(laps,     on=["session_key", "driver_number"], how="left")  # телеметрия кругов
    df = df.merge(weather,  on="session_key", how="left")
    df = df.merge(sessions[SESSION_COLS], on="session_key", how="left")
    df = df.merge(meetings[MEETING_COLS], on="meeting_key", how="left")
    return df


def build_quali_table(raw):
    # Квалификация — похожая структура, но без телеметрии кругов (laps_agg).
    results  = pd.read_csv(f"{raw}/quali_results.csv")
    drivers  = pd.read_csv(f"{raw}/drivers.csv")
    weather  = pd.read_csv(f"{raw}/weather.csv")
    sessions = pd.read_csv(f"{raw}/sessions.csv")
    meetings = pd.read_csv(f"{raw}/meetings.csv")

    df = results.merge(drivers, on=["session_key", "driver_number"], how="left")
    df = df.merge(weather,  on="session_key", how="left")
    df = df.merge(sessions[SESSION_COLS], on="session_key", how="left")
    df = df.merge(meetings[MEETING_COLS], on="meeting_key", how="left")
    return df


# Используем таблицы из памяти, если они уже есть; иначе собираем из raw/.
try:
    race
    quali
    print("Использую race / quali из памяти ноутбука")
except NameError:
    race  = build_race_table(RAW)
    quali = build_quali_table(RAW)
    print("Собрал race / quali из сырых файлов")

print("race:", race.shape, "| quali:", quali.shape)

Собрал race / quali из сырых файлов
race: (1397, 31) | quali: (1377, 23)


## Сортировка по дате

Перед расчётом любых признаков формы сортируем гонки по времени старта `date_start`.
Это обязательно: `shift`, `rolling` и `expanding` работают по порядку строк, поэтому
строки каждого пилота должны идти строго от старых гонок к новым.


In [3]:
# Дата в datetime + сортировка по времени. Признаки формы считаем по таблице race.
race["date_start"] = pd.to_datetime(race["date_start"])
race = race.sort_values("date_start").reset_index(drop=True)


# Вспомогательные функции расчёта по ПРОШЛЫМ гонкам.
# В обеих сначала shift(1) — это и есть защита от утечки:
# текущая гонка сдвигается и не попадает в своё же среднее.
def mean_last5_past(values):
    past = values.shift(1)                       # убираем текущую гонку
    return past.rolling(window=5, min_periods=1).mean()


def mean_all_past(values):
    past = values.shift(1)                       # убираем текущую гонку
    return past.expanding(min_periods=1).mean()


race.head()

,session_key,meeting_key,driver_number,position,number_of_laps,dnf,dns,dsq,gap_to_leader,points,...,air_temp_mean,track_temp_mean,humidity_mean,wind_speed_mean,rainfall_max,circuit_short_name,country_name,year,date_start,circuit_type
0,7953,1141,1,1.0,57.0,False,False,False,0,25.0,...,27.431677,31.011801,21.496894,0.68323,0,Sakhir,Bahrain,2023,2023-03-05 15:00:00+00:00,Permanent
1,7953,1141,81,NaN,13.0,True,False,False,NaN,0.0,...,27.431677,31.011801,21.496894,0.68323,0,Sakhir,Bahrain,2023,2023-03-05 15:00:00+00:00,Permanent
2,7953,1141,16,NaN,39.0,True,False,False,NaN,0.0,...,27.431677,31.011801,21.496894,0.68323,0,Sakhir,Bahrain,2023,2023-03-05 15:00:00+00:00,Permanent
3,7953,1141,31,NaN,41.0,True,False,False,NaN,0.0,...,27.431677,31.011801,21.496894,0.68323,0,Sakhir,Bahrain,2023,2023-03-05 15:00:00+00:00,Permanent
4,7953,1141,4,17.0,55.0,False,False,False,+2 LAPS,0.0,...,27.431677,31.011801,21.496894,0.68323,0,Sakhir,Bahrain,2023,2023-03-05 15:00:00+00:00,Permanent


### 1. `driver_form_5` — форма пилота за 5 последних гонок


In [4]:
# Средняя финишная позиция пилота за последние 5 ПРОШЛЫХ гонок.
# shift(1) внутри каждого пилота убирает текущую гонку -> нет утечки.
race["driver_form_5"] = race.groupby("driver_number")["position"].transform(mean_last5_past)

race[["date_start", "driver_number", "full_name", "position", "driver_form_5"]].head()

,date_start,driver_number,full_name,position,driver_form_5
0,2023-03-05 15:00:00+00:00,1,Max VERSTAPPEN,1.0,NaN
1,2023-03-05 15:00:00+00:00,81,Oscar PIASTRI,NaN,NaN
2,2023-03-05 15:00:00+00:00,16,Charles LECLERC,NaN,NaN
3,2023-03-05 15:00:00+00:00,31,Esteban OCON,NaN,NaN
4,2023-03-05 15:00:00+00:00,4,Lando NORRIS,17.0,NaN


### 2. `driver_form_all` — форма пилота за всю историю (expanding)


In [5]:
# Средняя позиция пилота по ВСЕМ прошлым гонкам (накопительное среднее).
# shift(1) снова исключает текущую гонку из расчёта.
race["driver_form_all"] = race.groupby("driver_number")["position"].transform(mean_all_past)

race[["date_start", "driver_number", "full_name", "position", "driver_form_all"]].head()

,date_start,driver_number,full_name,position,driver_form_all
0,2023-03-05 15:00:00+00:00,1,Max VERSTAPPEN,1.0,NaN
1,2023-03-05 15:00:00+00:00,81,Oscar PIASTRI,NaN,NaN
2,2023-03-05 15:00:00+00:00,16,Charles LECLERC,NaN,NaN
3,2023-03-05 15:00:00+00:00,31,Esteban OCON,NaN,NaN
4,2023-03-05 15:00:00+00:00,4,Lando NORRIS,17.0,NaN


### 3. `team_form_5` — форма команды за 5 последних результатов


In [6]:
# Средняя позиция КОМАНДЫ за 5 последних прошлых результатов команды.
# Группируем по team_name; shift(1) убирает текущий результат -> нет утечки.
race["team_form_5"] = race.groupby("team_name")["position"].transform(mean_last5_past)

race[["date_start", "team_name", "driver_number", "position", "team_form_5"]].head()

,date_start,team_name,driver_number,position,team_form_5
0,2023-03-05 15:00:00+00:00,Red Bull Racing,1,1.0,NaN
1,2023-03-05 15:00:00+00:00,McLaren,81,NaN,NaN
2,2023-03-05 15:00:00+00:00,Ferrari,16,NaN,NaN
3,2023-03-05 15:00:00+00:00,Alpine,31,NaN,NaN
4,2023-03-05 15:00:00+00:00,McLaren,4,17.0,NaN


### 4. `driver_track_history` — история пилота на этой трассе


In [7]:
# Средняя позиция пилота на ЭТОЙ ЖЕ трассе в прошлом (накопительное среднее).
# Группируем по паре (пилот, трасса); shift(1) исключает текущую гонку.
race["driver_track_history"] = (
    race.groupby(["driver_number", "circuit_short_name"])["position"]
        .transform(mean_all_past)
)

race[["date_start", "driver_number", "circuit_short_name", "position", "driver_track_history"]].head()

,date_start,driver_number,circuit_short_name,position,driver_track_history
0,2023-03-05 15:00:00+00:00,1,Sakhir,1.0,NaN
1,2023-03-05 15:00:00+00:00,81,Sakhir,NaN,NaN
2,2023-03-05 15:00:00+00:00,16,Sakhir,NaN,NaN
3,2023-03-05 15:00:00+00:00,31,Sakhir,NaN,NaN
4,2023-03-05 15:00:00+00:00,4,Sakhir,17.0,NaN


### 5. `driver_finish_rate` — доля завершённых гонок


In [8]:
# Доля прошлых гонок, которые пилот завершил без схода (dnf == False).
# Сначала признак "доехал" (1/0), потом среднее по прошлым гонкам.
# shift(1) внутри mean_all_past исключает текущую гонку.
race["completed"] = (~race["dnf"].astype(bool)).astype(int)   # 1 = доехал, 0 = сход
race["driver_finish_rate"] = race.groupby("driver_number")["completed"].transform(mean_all_past)

race[["date_start", "driver_number", "dnf", "completed", "driver_finish_rate"]].head()

,date_start,driver_number,dnf,completed,driver_finish_rate
0,2023-03-05 15:00:00+00:00,1,False,1,NaN
1,2023-03-05 15:00:00+00:00,81,True,0,NaN
2,2023-03-05 15:00:00+00:00,16,True,0,NaN
3,2023-03-05 15:00:00+00:00,31,True,0,NaN
4,2023-03-05 15:00:00+00:00,4,False,1,NaN


### 6. `driver_experience` — опыт пилота (число прошлых гонок)


In [9]:
# Сколько гонок пилот проехал ДО текущей.
# cumcount() нумерует гонки пилота с 0, поэтому это ровно число ПРОШЛЫХ гонок
# (текущая не считается) — отдельный shift тут не нужен.
race["driver_experience"] = race.groupby("driver_number").cumcount()

race[["date_start", "driver_number", "full_name", "driver_experience"]].head()

,date_start,driver_number,full_name,driver_experience
0,2023-03-05 15:00:00+00:00,1,Max VERSTAPPEN,0
1,2023-03-05 15:00:00+00:00,81,Oscar PIASTRI,0
2,2023-03-05 15:00:00+00:00,16,Charles LECLERC,0
3,2023-03-05 15:00:00+00:00,31,Esteban OCON,0
4,2023-03-05 15:00:00+00:00,4,Lando NORRIS,0


### 7. `driver_avg_speed_5` — средняя макс. скорость за 5 гонок (телеметрия)


In [10]:
# Средняя max_st_speed пилота за 5 последних ПРОШЛЫХ гонок.
# shift(1) внутри mean_last5_past исключает телеметрию текущей гонки -> нет утечки.
race["driver_avg_speed_5"] = race.groupby("driver_number")["max_st_speed"].transform(mean_last5_past)

race[["date_start", "driver_number", "max_st_speed", "driver_avg_speed_5"]].head()

,date_start,driver_number,max_st_speed,driver_avg_speed_5
0,2023-03-05 15:00:00+00:00,1,303.0,NaN
1,2023-03-05 15:00:00+00:00,81,327.0,NaN
2,2023-03-05 15:00:00+00:00,16,324.0,NaN
3,2023-03-05 15:00:00+00:00,31,329.0,NaN
4,2023-03-05 15:00:00+00:00,4,325.0,NaN


## Итог: новые признаки и пропуски (NaN)

Новые признаки и количество пропусков в каждом. NaN — это нормально:
у пилота/команды просто ещё не было прошлых гонок (или прошлых заездов на трассе),
по которым можно посчитать форму. Пока NaN НЕ заполняем.


In [11]:
new_columns = [
    "driver_form_5",
    "driver_form_all",
    "team_form_5",
    "driver_track_history",
    "driver_finish_rate",
    "driver_experience",
    "driver_avg_speed_5",
]

print("Добавлено новых признаков:", len(new_columns))
print()
print("Пропуски (NaN) по новым колонкам:")
print(race[new_columns].isna().sum())

Добавлено новых признаков: 7

Пропуски (NaN) по новым колонкам:
driver_form_5            38
driver_form_all          38
team_form_5              17
driver_track_history    702
driver_finish_rate       32
driver_experience         0
driver_avg_speed_5       33
dtype: int64
